# DHL Logistics RAG System
## Retrieval-Augmented Generation with Official DHL Documentation

**Project Highlights:**
- Built using official DHL documents (Express Terms, Customs Guide, eCommerce Terms)
- Complete RAG pipeline: Load → Split → Embed → Store → Retrieve → Generate
- LangGraph ReAct Agent for multi-tool orchestration
- Local deployment with zero API cost (Ollama + ChromaDB)
- Comprehensive evaluation framework with test dataset

**Tech Stack:** Python, LangChain, LangGraph, ChromaDB, Ollama, Pandas

---
## Phase 1: Offline Indexing Pipeline
### Load → Split → Embed → Store

In [108]:
# Step 1: Install Dependencies
import subprocess
import sys

packages = [
    "langchain",
    "langchain-community",
    "langchain-experimental",  # For SemanticChunker
    "langgraph",
    "langchain-ollama",
    "langchain-chroma",
    "langchain-text-splitters",
    "pypdf",
    "pandas",
    "matplotlib",
]

print("Installing dependencies...")
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ All dependencies installed successfully!")

Installing dependencies...
✅ All dependencies installed successfully!


In [109]:
# Step 2: Verify Ollama is Running
import requests

OLLAMA_BASE_URL = "http://localhost:11434"

print("🔍 Checking Ollama server...")
try:
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
    if response.status_code == 200:
        models = response.json()
        print(f"✅ Ollama is running!")
        print(f"   Available models: {[m['name'] for m in models['models']]}")
    else:
        print("❌ Ollama server not responding")
except requests.exceptions.ConnectionError:
    print("❌ Ollama is not running")
    print("\n📌 Setup Instructions:")
    print("   1. Download from: https://ollama.ai/download")
    print("   2. Run in terminal: ollama pull mistral")

🔍 Checking Ollama server...
✅ Ollama is running!
   Available models: ['mistral:latest']


---
### Step 3: Load Documents

The first step in RAG is loading source documents from PDF files.

```
PDF Files → PyPDFLoader → List of Document Objects
```

In [110]:
# Step 3: LOAD - Load DHL Official PDF Documents
from langchain_community.document_loaders import PyPDFLoader
import os

# PDF file paths
pdf_files = {
    "dhl_express_terms": "data/dhl_express_terms.pdf",      # DHL Express Terms & Conditions
    "dhl_customs_guide": "data/dhl_customs_guide.pdf",      # DHL US Customs Import Guide
    "dhl_ecommerce_terms": "data/dhl_ecommerce_terms.pdf",  # DHL eCommerce Terms
}

# Load all PDFs
all_documents = []

print("📄 Loading DHL official documents...\n")
for name, path in pdf_files.items():
    if os.path.exists(path):
        loader = PyPDFLoader(path)
        docs = loader.load()
        
        # Add source metadata to each document
        for doc in docs:
            doc.metadata["source_name"] = name
        
        all_documents.extend(docs)
        print(f"✅ {name}: {len(docs)} pages")
    else:
        print(f"❌ {name}: File not found")

print(f"\n📊 Total loaded: {len(all_documents)} pages")

📄 Loading DHL official documents...

✅ dhl_express_terms: 3 pages


Overwriting cache for 0 163


✅ dhl_customs_guide: 45 pages
✅ dhl_ecommerce_terms: 5 pages

📊 Total loaded: 53 pages


In [111]:
# Preview loaded document content
print("=" * 60)
print("Document Content Preview (First 500 characters of page 1)")
print("=" * 60)
print(all_documents[0].page_content[:500])
print("\n...\n")
print(f"Metadata: {all_documents[0].metadata}")

Document Content Preview (First 500 characters of page 1)
GENERAL TERMS AND CONDITIONS OF SALE DHL EXPRESS
Version of August, 28th 2024
ARTICLE 1 - PURPOSE AND SCOPE OF APPLICATION
The purpose of these General Terms and Conditions of Sale (hereinafter “GTCS”) is to establish the 
terms applicable to the services provided, in any capacity whatsoever, by DHL International Express 
(France) SAS, a French corporation (société par actions simplifiée) with share capital of 19 347 230 
euros, having its registered office at 53 avenue Jean-Jaurès, 93350 Le Bou

...

Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.3 (Macintosh)', 'creationdate': '2024-08-20T08:54:17+02:00', 'moddate': '2024-08-20T08:54:18+02:00', 'trapped': '/False', 'source': 'data/dhl_express_terms.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_name': 'dhl_express_terms'}


---
### Step 4: Semantic Chunking

**Why Semantic Chunking instead of fixed-size chunking?**

| Method | How it works | Pros | Cons |
|--------|-------------|------|------|
| **Fixed-size** | Split every N characters | Fast, simple | Cuts sentences, ignores meaning |
| **Semantic** | Split when meaning changes | Preserves context, smarter boundaries | Slower (needs embeddings) |

**How Semantic Chunking works:**
```
Sentence 1: "DHL Express delivers worldwide."     ─┐
Sentence 2: "We offer next-day delivery."         ─┤ Similar → Same chunk
Sentence 3: "Tracking is available online."       ─┘
                                                    ← Semantic break detected
Sentence 4: "Prohibited items include weapons."   ─┐
Sentence 5: "Explosives are not allowed."         ─┤ Similar → New chunk
```

In [112]:
# Step 4: SPLIT - Semantic Chunking
from langchain_experimental.text_splitter import SemanticChunker
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings

# Initialize embeddings for semantic chunking
OLLAMA_BASE_URL = "http://localhost:11434"
chunking_embeddings = OllamaEmbeddings(
    model="mistral",
    base_url=OLLAMA_BASE_URL
)

# Method 1: Semantic Chunker (recommended)
# Splits based on semantic similarity between sentences
semantic_splitter = SemanticChunker(
    embeddings=chunking_embeddings,
    breakpoint_threshold_type="percentile",  # Options: "percentile", "standard_deviation", "interquartile"
    breakpoint_threshold_amount=85,          # Higher = fewer, larger chunks
)

print("🧠 Using Semantic Chunking...")
print("   Breakpoint type: percentile")
print("   Threshold: 85 (meaning: split when similarity drops below 85th percentile)")

# Combine all document text for semantic chunking
# SemanticChunker works on text, so we need to process and preserve metadata
chunks = []
for doc in all_documents:
    try:
        # Split this document semantically
        doc_chunks = semantic_splitter.create_documents(
            texts=[doc.page_content],
            metadatas=[doc.metadata]
        )
        chunks.extend(doc_chunks)
    except Exception as e:
        # Fallback to simple splitting if semantic fails for this doc
        print(f"   ⚠️ Fallback for {doc.metadata.get('source_name', 'unknown')}: {e}")
        fallback_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        doc_chunks = fallback_splitter.split_documents([doc])
        chunks.extend(doc_chunks)

print(f"\n📊 Semantic Chunking Results:")
print(f"   Original documents: {len(all_documents)} pages")
print(f"   After chunking: {len(chunks)} chunks")
print(f"   Average chunks per page: {len(chunks)/len(all_documents):.1f}")

🧠 Using Semantic Chunking...
   Breakpoint type: percentile
   Threshold: 85 (meaning: split when similarity drops below 85th percentile)

📊 Semantic Chunking Results:
   Original documents: 53 pages
   After chunking: 200 chunks
   Average chunks per page: 3.8


In [113]:
# Preview semantic chunking results - compare chunk sizes
print("=" * 60)
print("Semantic Chunk Analysis")
print("=" * 60)

# Analyze chunk size distribution
chunk_sizes = [len(chunk.page_content) for chunk in chunks]
print(f"\n📏 Chunk Size Statistics:")
print(f"   Min: {min(chunk_sizes)} chars")
print(f"   Max: {max(chunk_sizes)} chars")
print(f"   Average: {sum(chunk_sizes)/len(chunk_sizes):.0f} chars")
print(f"   Median: {sorted(chunk_sizes)[len(chunk_sizes)//2]} chars")

print("\n📄 Sample Chunks (showing semantic boundaries):")
print("-" * 60)

for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ({len(chunk.page_content)} chars) ---")
    print(f"Source: {chunk.metadata.get('source_name', 'unknown')}")
    # Show first and last 150 chars to see boundaries
    content = chunk.page_content
    if len(content) > 350:
        print(f"Start: {content[:150]}...")
        print(f"...End: ...{content[-150:]}")
    else:
        print(f"Content: {content}")

Semantic Chunk Analysis

📏 Chunk Size Statistics:
   Min: 2 chars
   Max: 4772 chars
   Average: 682 chars
   Median: 397 chars

📄 Sample Chunks (showing semantic boundaries):
------------------------------------------------------------

--- Chunk 1 (587 chars) ---
Source: dhl_express_terms
Start: GENERAL TERMS AND CONDITIONS OF SALE DHL EXPRESS
Version of August, 28th 2024
ARTICLE 1 - PURPOSE AND SCOPE OF APPLICATION
The purpose of these Genera...
...End: ...ng its registered office at 53 avenue Jean-Jaurès, 93350 Le Bourget, registered in Bobigny 
under number 494 956 774 thereinafter designated as “DHL”.

--- Chunk 2 (253 chars) ---
Source: dhl_express_terms
Content: By using DHL services, you accept the following conditions:
1. Where applicable, the Special Terms and Conditions applicable to the services in question 
(applicable only if you are a professional);
2. Our General Terms and Conditions of Sale (GTCS);
3.

--- Chunk 3 (806 chars) ---
Source: dhl_express_terms
Start: Our M

---
### Step 5: Generate Embeddings

Convert text into numerical vectors. Semantically similar text will have similar vectors.

```
"DHL delivers in 24 hours" → [0.12, -0.34, 0.56, ..., 0.08] (768 dimensions)
"Express shipping next day" → [0.11, -0.32, 0.55, ..., 0.09]  ← Similar meaning = Similar vectors
```

In [114]:
# Step 5: EMBED - Generate Vector Embeddings
from langchain_ollama import OllamaEmbeddings

# Initialize embeddings using local Ollama model
embeddings = OllamaEmbeddings(
    model="mistral",
    base_url=OLLAMA_BASE_URL
)

# Test embedding generation
test_text = "What is DHL delivery time?"
test_embedding = embeddings.embed_query(test_text)

print(f"✅ Embedding model initialized successfully")
print(f"\nTest text: '{test_text}'")
print(f"Vector dimensions: {len(test_embedding)}")
print(f"First 10 values: {test_embedding[:10]}")

✅ Embedding model initialized successfully

Test text: 'What is DHL delivery time?'
Vector dimensions: 4096
First 10 values: [0.023890072, 0.0021159868, -0.007787771, -0.0035162994, 0.002072989, -0.022405555, 0.018966993, 0.008230567, 0.022225574, -0.013503646]


---
### Step 6: Store in Vector Database

ChromaDB storage structure:
```
┌──────┬─────────────────────────┬──────────────────────┐
│  ID  │  Vector (768 dims)      │  Document            │
├──────┼─────────────────────────┼──────────────────────┤
│  1   │  [0.12, -0.34, ...]     │  "DHL Express..."    │
│  2   │  [0.08, -0.21, ...]     │  "Customs guide..."  │
└──────┴─────────────────────────┴──────────────────────┘
```

In [115]:
# Step 6: STORE - Create Vector Database
from langchain_chroma import Chroma
import chromadb

# Use in-memory client to avoid persistence issues
# For production, you can switch back to PersistentClient after fixing chromadb version
chroma_client = chromadb.Client()

print("🚀 Creating vector database (this may take a few minutes)...")
print(f"   Processing {len(chunks)} chunks...")

# Create Chroma vector store with in-memory client
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="dhl_knowledge",
    client=chroma_client  # Use in-memory client instead of persist_directory
)

print(f"\n✅ Vector database created successfully!")
print(f"   Storage: In-memory")
print(f"   Document count: {len(chunks)} chunks")

🚀 Creating vector database (this may take a few minutes)...
   Processing 200 chunks...

✅ Vector database created successfully!
   Storage: In-memory
   Document count: 200 chunks


---
## Phase 2: Online Query Pipeline
### Retrieve → Generate

---
### Step 7: Create Retriever

User query → Vectorize → Find most similar documents in database

```
User: "What are prohibited items?"
        ↓ Vectorize
    [0.14, -0.33, 0.54, ...]
        ↓ Cosine similarity search
    Return Top-K most relevant chunks
```

In [116]:
# Step 7: RETRIEVE - Create Retriever

# Create retriever that returns top 10 most similar documents
# More documents = higher recall, but may include less relevant content
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}  # Increased from 5 to 10 for better keyword coverage
)

print("✅ Retriever created successfully")
print("   Search type: similarity (cosine)")
print("   Results returned: Top 10")

✅ Retriever created successfully
   Search type: similarity (cosine)
   Results returned: Top 10


In [117]:
# Test retrieval performance
test_queries = [
    "What items are prohibited for shipping?",
    "How do I file a claim for lost package?",
    "What are the customs clearance procedures?"
]

print("=" * 60)
print("Retrieval Test")
print("=" * 60)

for query in test_queries:
    print(f"\n❓ Query: {query}")
    docs = retriever.invoke(query)
    print(f"📄 Retrieved {len(docs)} relevant documents:")
    for i, doc in enumerate(docs[:2]):  # Show only top 2
        print(f"   [{i+1}] {doc.page_content[:150]}...")

Retrieval Test

❓ Query: What items are prohibited for shipping?
📄 Retrieved 10 relevant documents:
   [1] These Restricted Shipments are subject to authorization by DHL after examination....
   [2] These Restricted Shipments are subject to authorization by DHL after examination....

❓ Query: How do I file a claim for lost package?
📄 Retrieved 10 relevant documents:
   [1] These Restricted Shipments are subject to authorization by DHL after examination....
   [2] These Restricted Shipments are subject to authorization by DHL after examination....

❓ Query: What are the customs clearance procedures?
📄 Retrieved 10 relevant documents:
   [1] In some instances this is related 
to registration that must be carried out prior to 
shipping, in addition to what is required to be 
declared on air...
   [2] In some instances this is related 
to registration that must be carried out prior to 
shipping, in addition to what is required to be 
declared on air...


---
### Step 8: Build LangGraph Agent

LangGraph ReAct Agent = Reasoning + Acting

```
User Question
    ↓
Agent Thinks: Which tool should I use?
    ↓
Execute Tool (search knowledge / check status / estimate cost)
    ↓
LLM generates answer based on tool results
```

In [118]:
# Step 8: Define Agent Tools
from langchain_core.tools import tool
import pandas as pd
from datetime import datetime, timedelta

# Tool 1: Search DHL Knowledge Base
@tool
def search_dhl_knowledge(query: str) -> str:
    """Search DHL official documentation for information about shipping, customs, terms, etc."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant information found."
    
    context = "\n\n".join([
        f"[Source: {doc.metadata.get('source_name', 'unknown')}]\n{doc.page_content}" 
        for doc in docs[:3]
    ])
    return f"DHL Documentation:\n{context}"

# Tool 2: Check Shipment Status (Simulated Data)
shipments_data = pd.DataFrame({
    "tracking_id": ["DHL001", "DHL002", "DHL003", "DHL004", "DHL005"],
    "origin": ["New York", "Los Angeles", "Chicago", "Houston", "Miami"],
    "destination": ["London", "Tokyo", "Berlin", "Paris", "Sydney"],
    "status": ["IN_TRANSIT", "OUT_FOR_DELIVERY", "DELIVERED", "CUSTOMS_HOLD", "EXCEPTION"],
    "weight_kg": [2.5, 5.0, 1.2, 8.3, 3.7],
    "ship_date": [datetime.now() - timedelta(days=i) for i in range(2, 7)],
    "eta": [datetime.now() + timedelta(days=i) for i in range(1, 6)],
})

@tool
def check_shipment(tracking_id: str) -> str:
    """Check the status of a DHL shipment by tracking ID."""
    shipment = shipments_data[shipments_data['tracking_id'] == tracking_id.upper()]
    
    if shipment.empty:
        return f"Shipment {tracking_id} not found."
    
    s = shipment.iloc[0]
    return f"""
📦 Shipment: {s['tracking_id']}
Route: {s['origin']} → {s['destination']}
Status: {s['status']}
Weight: {s['weight_kg']} kg
Shipped: {s['ship_date'].strftime('%Y-%m-%d')}
ETA: {s['eta'].strftime('%Y-%m-%d')}
""".strip()

# Tool 3: Estimate Shipping Cost
@tool  
def estimate_shipping_cost(origin: str, destination: str, weight_kg: float) -> str:
    """Estimate DHL shipping cost based on origin, destination and weight."""
    base_rate = 25.0
    weight_rate = 5.0  # per kg
    international_surcharge = 15.0 if origin != destination else 0
    
    total = base_rate + (weight_kg * weight_rate) + international_surcharge
    
    return f"""
💰 Shipping Cost Estimate:
Route: {origin} → {destination}
Weight: {weight_kg} kg
Base Rate: ${base_rate:.2f}
Weight Charge: ${weight_kg * weight_rate:.2f}
International: ${international_surcharge:.2f}
━━━━━━━━━━━━━━━━━━
Total: ${total:.2f}
""".strip()

# Tool list
tools = [search_dhl_knowledge, check_shipment, estimate_shipping_cost]

print(f"✅ Created {len(tools)} tools:")
for t in tools:
    print(f"   - {t.name}: {t.description}")

✅ Created 3 tools:
   - search_dhl_knowledge: Search DHL official documentation for information about shipping, customs, terms, etc.
   - check_shipment: Check the status of a DHL shipment by tracking ID.
   - estimate_shipping_cost: Estimate DHL shipping cost based on origin, destination and weight.


In [119]:
# Step 9: Create LangGraph ReAct Agent
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

# Initialize LLM
llm = ChatOllama(
    model="mistral",
    base_url=OLLAMA_BASE_URL,
    temperature=0.3  # Lower temperature = more consistent output
)

# Create ReAct Agent
agent = create_react_agent(llm, tools)

print("✅ LangGraph ReAct Agent created successfully")
print("\n📋 Agent Configuration:")
print(f"   LLM: Mistral (Ollama)")
print(f"   Tools: {len(tools)}")
print(f"   Framework: LangGraph ReAct")

✅ LangGraph ReAct Agent created successfully

📋 Agent Configuration:
   LLM: Mistral (Ollama)
   Tools: 3
   Framework: LangGraph ReAct


/var/folders/7n/rym0b3rj47997yj8_k0ph8gh0000gn/T/ipykernel_62131/1188501646.py:13: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


In [120]:
# Query function wrapper
def ask_dhl(question: str) -> str:
    """
    Ask the DHL RAG Agent a question.
    
    Example questions:
    - "What items cannot be shipped via DHL?"
    - "Check status of DHL001"
    - "How much to ship 5kg from New York to London?"
    """
    print(f"\n{'='*60}")
    print(f"❓ Question: {question}")
    print(f"{'='*60}")
    
    response = agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })
    
    answer = response["messages"][-1].content
    print(f"\n🤖 Answer:\n{answer}")
    return answer

---
## Test the RAG System

In [121]:
# Test 1: Knowledge Base Search
ask_dhl("What items are prohibited from shipping with DHL?")


❓ Question: What items are prohibited from shipping with DHL?

🤖 Answer:
 To get the information about the items that are prohibited from shipping with DHL, we can use the `search_dhl_knowledge` function. Here's how you can do it:

```javascript
[{"name":"search_dhl_knowledge","arguments":{"query": "What items are prohibited from shipping with DHL?"}}]
```

This will search the DHL official documentation for information about items that are prohibited from shipping.


' To get the information about the items that are prohibited from shipping with DHL, we can use the `search_dhl_knowledge` function. Here\'s how you can do it:\n\n```javascript\n[{"name":"search_dhl_knowledge","arguments":{"query": "What items are prohibited from shipping with DHL?"}}]\n```\n\nThis will search the DHL official documentation for information about items that are prohibited from shipping.'

In [122]:
# Test 2: Shipment Status Query
ask_dhl("What is the status of shipment DHL004?")


❓ Question: What is the status of shipment DHL004?

🤖 Answer:
 The status of shipment DHL004 is "CUSTOMS_HOLD". It was shipped on February 2nd, 2026 from Houston to Paris. The estimated arrival date is February 11th, 2026. The weight of the shipment is 8.3 kg.


' The status of shipment DHL004 is "CUSTOMS_HOLD". It was shipped on February 2nd, 2026 from Houston to Paris. The estimated arrival date is February 11th, 2026. The weight of the shipment is 8.3 kg.'

In [123]:
# Test 3: Cost Estimation
ask_dhl("How much would it cost to ship a 3kg package from New York to London?")


❓ Question: How much would it cost to ship a 3kg package from New York to London?

🤖 Answer:
 The estimated shipping cost for a 3kg package from New York to London is approximately $55.00. This includes base rate, weight charge, and international charges.


' The estimated shipping cost for a 3kg package from New York to London is approximately $55.00. This includes base rate, weight charge, and international charges.'

In [124]:
# Test 4: Complex Query (requires knowledge base search)
ask_dhl("What should I do if my shipment is held at customs? What documents do I need?")


❓ Question: What should I do if my shipment is held at customs? What documents do I need?

🤖 Answer:
 To find out the specific documents required for a shipment that is being held at customs, you can use the `search_dhl_knowledge` function:

```javascript
[{"name":"search_dhl_knowledge","arguments":{"query":"What documents are required for a shipment held at customs?"}}]
```

This will search DHL's official documentation and return the necessary documents needed for a shipment being held at customs.


' To find out the specific documents required for a shipment that is being held at customs, you can use the `search_dhl_knowledge` function:\n\n```javascript\n[{"name":"search_dhl_knowledge","arguments":{"query":"What documents are required for a shipment held at customs?"}}]\n```\n\nThis will search DHL\'s official documentation and return the necessary documents needed for a shipment being held at customs.'

---
## Phase 3: Evaluation Framework

Two dimensions of RAG evaluation:
1. **Retrieval Evaluation** - Are the retrieved documents relevant?
2. **Generation Evaluation** - Is the generated answer correct?

### Metrics:
| Metric | Description |
|--------|-------------|
| **Recall@K** | Proportion of queries where correct document is in top K results |
| **MRR** | Mean Reciprocal Rank - average of 1/rank of first correct result |
| **Relevance** | LLM-judged: Is the answer relevant to the question? (1-5) |
| **Completeness** | LLM-judged: Does the answer fully address the question? (1-5) |
| **Accuracy** | LLM-judged: Does the answer seem factually correct? (1-5) |

In [125]:
# Step 10: Create Test Dataset (Ground Truth)
# Each test case contains: question, expected keywords, expected source document, category
# Keywords are more generalized for flexible matching

test_dataset = [
    {
        "id": 1,
        "question": "What items are prohibited from shipping with DHL?",
        "expected_keywords": ["prohibit", "dangerous", "goods", "ship", "hazard", "item"],
        "expected_source": "dhl_express_terms",
        "category": "prohibited_items"
    },
    {
        "id": 2,
        "question": "What is DHL's liability limit for lost packages?",
        "expected_keywords": ["liab", "limit", "loss", "value", "amount", "damage"],
        "expected_source": "dhl_express_terms",
        "category": "liability"
    },
    {
        "id": 3,
        "question": "How do I file a claim for a damaged shipment?",
        "expected_keywords": ["claim", "damage", "notify", "day", "write", "request"],
        "expected_source": "dhl_express_terms",
        "category": "claims"
    },
    {
        "id": 4,
        "question": "What documents are required for customs clearance?",
        "expected_keywords": ["custom", "document", "invoice", "clear", "import", "declare"],
        "expected_source": "dhl_customs_guide",
        "category": "customs"
    },
    {
        "id": 5,
        "question": "What is the process for importing goods to the US?",
        "expected_keywords": ["import", "custom", "duty", "entry", "broker", "goods"],
        "expected_source": "dhl_customs_guide",
        "category": "customs"
    },
    {
        "id": 6,
        "question": "What are the delivery time guarantees for DHL Express?",
        "expected_keywords": ["deliver", "time", "service", "express", "business", "day"],
        "expected_source": "dhl_express_terms",
        "category": "delivery"
    },
    {
        "id": 7,
        "question": "How does DHL handle shipments that cannot be delivered?",
        "expected_keywords": ["deliver", "return", "address", "contact", "attempt", "shipment"],
        "expected_source": "dhl_express_terms",
        "category": "delivery"
    },
    {
        "id": 8,
        "question": "What are the packaging requirements for DHL shipments?",
        "expected_keywords": ["pack", "shipment", "box", "label", "secure", "content"],
        "expected_source": "dhl_express_terms",
        "category": "packaging"
    },
    {
        "id": 9,
        "question": "What duties and taxes apply to international shipments?",
        "expected_keywords": ["dut", "tax", "custom", "import", "charge", "pay"],
        "expected_source": "dhl_customs_guide",
        "category": "customs"
    },
    {
        "id": 10,
        "question": "What is DHL's privacy policy regarding shipment data?",
        "expected_keywords": ["privacy", "data", "information", "personal", "policy", "use"],
        "expected_source": "dhl_express_terms",
        "category": "privacy"
    },
]

print(f"✅ Test dataset created: {len(test_dataset)} test cases")
print("\n📋 Category distribution:")
categories = {}
for item in test_dataset:
    cat = item["category"]
    categories[cat] = categories.get(cat, 0) + 1
for cat, count in categories.items():
    print(f"   - {cat}: {count} questions")

✅ Test dataset created: 10 test cases

📋 Category distribution:
   - prohibited_items: 1 questions
   - liability: 1 questions
   - claims: 1 questions
   - customs: 3 questions
   - delivery: 2 questions
   - packaging: 1 questions
   - privacy: 1 questions


In [ ]:
# Step 11: Retrieval Evaluation

def evaluate_retrieval(test_data, retriever, k=10):
    """
    Evaluate retrieval quality.
    
    Metrics:
    - Recall@K: Proportion of queries with correct source in top K
    - MRR: Mean Reciprocal Rank (higher = correct doc ranks higher)
    """
    results = []
    
    for item in test_data:
        question = item["question"]
        expected_source = item["expected_source"]
        
        # Retrieve documents
        retrieved_docs = retriever.invoke(question)
        
        # Check source match
        retrieved_sources = [doc.metadata.get("source_name", "") for doc in retrieved_docs]
        source_hit = expected_source in retrieved_sources
        
        # Calculate source rank (for MRR)
        source_rank = None
        for i, source in enumerate(retrieved_sources):
            if source == expected_source:
                source_rank = i + 1
                break
        
        results.append({
            "id": item["id"],
            "question": question[:50] + "...",
            "source_hit": source_hit,
            "source_rank": source_rank,
            "category": item["category"]
        })
    
    # Calculate overall metrics
    recall_at_k = sum(1 for r in results if r["source_hit"]) / len(results)
    mrr = sum(1/r["source_rank"] for r in results if r["source_rank"]) / len(results)
    
    return {
        "results": results,
        "metrics": {
            "Recall@K": recall_at_k,
            "MRR": mrr
        },
        "k": k
    }

# Run retrieval evaluation
K = 10
print(f"🔍 Running retrieval evaluation (k={K})...")
print("=" * 60)

retrieval_eval = evaluate_retrieval(test_dataset, retriever, k=K)

print(f"\n📊 Retrieval Evaluation Results:")
print(f"   Recall@{K}: {retrieval_eval['metrics']['Recall@K']:.1%}")
print(f"   MRR: {retrieval_eval['metrics']['MRR']:.3f}")

print("\n📋 Detailed Results:")
for r in retrieval_eval["results"]:
    status = "✅" if r["source_hit"] else "❌"
    rank_info = f"rank {r['source_rank']}" if r["source_rank"] else "not found"
    print(f"   {status} Q{r['id']}: {r['question']} ({rank_info})")

In [ ]:
# Step 12: Generation Evaluation (LLM-as-Judge)

def evaluate_answer_with_llm(question: str, answer: str) -> dict:
    """
    Use LLM to evaluate answer quality.
    
    Scoring dimensions:
    1. Relevance (1-5): Is the answer relevant to the question?
    2. Completeness (1-5): Does the answer fully address the question?
    3. Accuracy (1-5): Does the answer seem factually correct?
    """
    eval_prompt = f"""You are an evaluation assistant. Rate the following answer on a scale of 1-5 for each criterion.

Question: {question}

Answer: {answer}

Rate the answer (respond with ONLY three numbers separated by commas, e.g., "4,3,5"):
1. Relevance (1-5): Is the answer relevant to the question?
2. Completeness (1-5): Does the answer fully address the question?
3. Accuracy (1-5): Does the answer seem factually correct?

Scores:"""

    try:
        response = llm.invoke(eval_prompt)
        scores_text = response.content.strip()
        
        # Parse scores
        import re
        numbers = re.findall(r'\d+', scores_text)
        if len(numbers) >= 3:
            relevance = min(int(numbers[0]), 5)
            completeness = min(int(numbers[1]), 5)
            accuracy = min(int(numbers[2]), 5)
        else:
            relevance = completeness = accuracy = 3  # Default scores
            
        return {
            "relevance": relevance,
            "completeness": completeness,
            "accuracy": accuracy,
            "average": (relevance + completeness + accuracy) / 3
        }
    except Exception as e:
        return {"relevance": 0, "completeness": 0, "accuracy": 0, "average": 0, "error": str(e)}


def run_full_evaluation(test_data, agent, retriever, num_samples=5):
    """
    Run complete RAG evaluation (retrieval + generation).
    """
    results = []
    
    print(f"🧪 Evaluating {num_samples} test cases...")
    print("=" * 60)
    
    for i, item in enumerate(test_data[:num_samples]):
        print(f"\n[{i+1}/{num_samples}] Evaluating: {item['question'][:50]}...")
        
        # 1. Get Agent answer
        try:
            response = agent.invoke({
                "messages": [{"role": "user", "content": item["question"]}]
            })
            answer = response["messages"][-1].content
        except Exception as e:
            answer = f"Error: {e}"
        
        # 2. Retrieval evaluation
        retrieved_docs = retriever.invoke(item["question"])
        retrieved_sources = [doc.metadata.get("source_name", "") for doc in retrieved_docs]
        source_hit = item["expected_source"] in retrieved_sources
        
        # 3. LLM-as-Judge evaluation
        llm_scores = evaluate_answer_with_llm(item["question"], answer)
        
        results.append({
            "id": item["id"],
            "question": item["question"],
            "answer": answer[:200] + "..." if len(answer) > 200 else answer,
            "source_hit": source_hit,
            "llm_scores": llm_scores,
            "category": item["category"]
        })
        
        print(f"   Source: {'✅' if source_hit else '❌'} | LLM Score: {llm_scores['average']:.1f}/5")
    
    return results


# Run full evaluation (using first 5 test cases to save time)
print("\n🚀 Starting Full RAG Evaluation")
eval_results = run_full_evaluation(test_dataset, agent, retriever, num_samples=5)

In [ ]:
# Step 13: Generate Evaluation Report

def generate_evaluation_report(eval_results, retrieval_eval):
    """Generate comprehensive evaluation report."""
    
    retrieval_metrics = retrieval_eval["metrics"]
    k = retrieval_eval.get("k", 10)
    
    print("\n" + "=" * 60)
    print("📊 RAG SYSTEM EVALUATION REPORT")
    print("=" * 60)
    
    # 1. Retrieval Metrics
    print("\n【1. Retrieval Evaluation】")
    print(f"   Recall@{k}:       {retrieval_metrics['Recall@K']:.1%}")
    print(f"   MRR:             {retrieval_metrics['MRR']:.3f}")
    
    # 2. Generation Metrics
    print("\n【2. Generation Evaluation (LLM-as-Judge)】")
    
    avg_relevance = sum(r["llm_scores"]["relevance"] for r in eval_results) / len(eval_results)
    avg_completeness = sum(r["llm_scores"]["completeness"] for r in eval_results) / len(eval_results)
    avg_accuracy = sum(r["llm_scores"]["accuracy"] for r in eval_results) / len(eval_results)
    avg_overall = sum(r["llm_scores"]["average"] for r in eval_results) / len(eval_results)
    
    print(f"   Relevance:       {avg_relevance:.1f}/5")
    print(f"   Completeness:    {avg_completeness:.1f}/5")
    print(f"   Accuracy:        {avg_accuracy:.1f}/5")
    print(f"   Overall Score:   {avg_overall:.1f}/5 ({avg_overall/5*100:.0f}%)")
    
    # 3. Category Analysis
    print("\n【3. Category Analysis】")
    categories = {}
    for r in eval_results:
        cat = r["category"]
        if cat not in categories:
            categories[cat] = []
        categories[cat].append(r["llm_scores"]["average"])
    
    for cat, scores in categories.items():
        avg = sum(scores) / len(scores)
        print(f"   {cat}: {avg:.1f}/5")
    
    # 4. Summary
    print("\n【4. Summary】")
    print("=" * 60)
    
    overall_score = (retrieval_metrics['Recall@K'] + avg_overall/5) / 2 * 100
    
    if overall_score >= 80:
        grade = "A (Excellent)"
    elif overall_score >= 70:
        grade = "B (Good)"
    elif overall_score >= 60:
        grade = "C (Satisfactory)"
    else:
        grade = "D (Needs Improvement)"
    
    print(f"   Overall Score: {overall_score:.0f}/100 - {grade}")
    print(f"   Test Samples: {len(eval_results)}")
    print(f"   Knowledge Base: DHL Official Docs (Express, Customs, eCommerce)")
    
    return {
        "retrieval": retrieval_metrics,
        "generation": {
            "relevance": avg_relevance,
            "completeness": avg_completeness,
            "accuracy": avg_accuracy,
            "overall": avg_overall
        },
        "overall_score": overall_score,
        "grade": grade,
        "k": k
    }

# Generate report
final_report = generate_evaluation_report(eval_results, retrieval_eval)

In [ ]:
# Step 14: Visualize Evaluation Results

import matplotlib.pyplot as plt

def plot_evaluation_results(eval_results, retrieval_eval):
    """Visualize evaluation results."""
    
    retrieval_metrics = retrieval_eval["metrics"]
    k = retrieval_eval.get("k", 10)
    
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    # Chart 1: Retrieval Metrics
    ax1 = axes[0]
    metrics = [f"Recall@{k}", "MRR"]
    values = [retrieval_metrics["Recall@K"], retrieval_metrics["MRR"]]
    colors = ["#2ecc71" if v >= 0.7 else "#f39c12" if v >= 0.5 else "#e74c3c" for v in values]
    bars = ax1.bar(metrics, values, color=colors)
    ax1.set_ylim(0, 1)
    ax1.set_title("Retrieval Metrics", fontsize=12, fontweight="bold")
    ax1.set_ylabel("Score")
    for bar, v in zip(bars, values):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f"{v:.0%}", ha="center", fontsize=10)
    
    # Chart 2: Generation Quality Scores (LLM-as-Judge)
    ax2 = axes[1]
    gen_metrics = ["Relevance", "Completeness", "Accuracy"]
    gen_values = [
        sum(r["llm_scores"]["relevance"] for r in eval_results) / len(eval_results),
        sum(r["llm_scores"]["completeness"] for r in eval_results) / len(eval_results),
        sum(r["llm_scores"]["accuracy"] for r in eval_results) / len(eval_results),
    ]
    colors = ["#3498db"] * 3
    bars = ax2.bar(gen_metrics, gen_values, color=colors)
    ax2.set_ylim(0, 5)
    ax2.set_title("Generation Quality (LLM-as-Judge)", fontsize=12, fontweight="bold")
    ax2.set_ylabel("Score (1-5)")
    for bar, v in zip(bars, gen_values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                f"{v:.1f}", ha="center", fontsize=10)
    
    # Chart 3: Score by Question
    ax3 = axes[2]
    questions = [f"Q{r['id']}" for r in eval_results]
    scores = [r["llm_scores"]["average"] for r in eval_results]
    colors = ["#2ecc71" if s >= 4 else "#f39c12" if s >= 3 else "#e74c3c" for s in scores]
    bars = ax3.bar(questions, scores, color=colors)
    ax3.set_ylim(0, 5)
    ax3.set_title("Score by Question", fontsize=12, fontweight="bold")
    ax3.set_ylabel("Average Score (1-5)")
    ax3.axhline(y=3.5, color="gray", linestyle="--", alpha=0.5)
    
    plt.tight_layout()
    plt.savefig("evaluation_results.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\n📈 Chart saved: evaluation_results.png")

# Generate visualization
try:
    plot_evaluation_results(eval_results, retrieval_eval)
except Exception as e:
    print(f"⚠️ Visualization failed: {e}")

---
## System Architecture Summary

```
┌─────────────────────────────────────────────────────────────┐
│                  OFFLINE PHASE (Indexing)                   │
├─────────────────────────────────────────────────────────────┤
│  DHL PDFs → Load → Split → Embed → Store (ChromaDB)        │
└─────────────────────────────────────────────────────────────┘
                              ↓
┌─────────────────────────────────────────────────────────────┐
│                  ONLINE PHASE (Querying)                    │
├─────────────────────────────────────────────────────────────┤
│  User Question                                              │
│      ↓                                                      │
│  LangGraph ReAct Agent                                      │
│      ├── Tool 1: search_dhl_knowledge (Vector Retrieval)    │
│      ├── Tool 2: check_shipment (Status Query)              │
│      └── Tool 3: estimate_shipping_cost (Cost Calculator)   │
│      ↓                                                      │
│  Mistral LLM Generates Response                             │
└─────────────────────────────────────────────────────────────┘
                              ↓
┌─────────────────────────────────────────────────────────────┐
│                  EVALUATION PHASE                           │
├─────────────────────────────────────────────────────────────┤
│  Test Dataset (10 cases) → Retrieval Eval → Generation Eval │
│  Metrics: Recall@K, MRR, LLM-as-Judge Scoring               │
└─────────────────────────────────────────────────────────────┘
```